# Gradient Boosting (GBM)

Friedman GBM (2001): https://projecteuclid.org/journals/annals-of-statistics/volume-29/issue-5/Greedy-function-approximation-A-gradient-boosting-machine/10.1214/aos/1013203451.full




XGBoost (2016): https://arxiv.org/abs/1603.02754



LightGBM (2017): https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html




CatBoost (2018): https://arxiv.org/abs/1706.09516




NGBoost (2020): https://arxiv.org/abs/1910.03225



# Gradient Boosting

In [1]:
# Arabic Sentiment Analysis — Gradient Boosting (Friedman / sklearn)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH = "/content/train_clean.csv"
VAL_PATH   = "/content/val_clean.csv"
TEST_PATH  = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# sklearn GBM requires dense arrays
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)]).toarray()
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)]).toarray()
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)]).toarray()

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — Gradient Boosting (Friedman)
# Classic gradient boosting with decision trees as weak learners.
# Uses 2nd-order Taylor expansion of the loss function.
# `subsample < 1` = Stochastic Gradient Boosting → reduces variance.
print("=" * 60)
print("  🚀 Training: Gradient Boosting (Friedman / sklearn)")
print("=" * 60)

clf = GradientBoostingClassifier(
    n_estimators  = 300,
    learning_rate = 0.05,   # shrinkage — lower = more robust, needs more trees
    max_depth     = 5,
    subsample     = 0.8,    # stochastic GB
    max_features  = "sqrt", # random feature subset per split
    random_state  = RANDOM_STATE,
)

clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Gradient Boosting  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 10,264
Final feature shape (train) : (5067, 10266)

Classes : [0 1]  →  [0, 1]

  🚀 Training: Gradient Boosting (Friedman / sklearn)

────────────────────────────────────────────────────────────
  Gradient Boosting  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7514
  Precision : 0.7556
  Recall    : 0.7514
  F1-Score  : 0.7503

  Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.82      0.77       543
           1       0.79      0.69      0.73       543

    accuracy                           0.75      1086
   macro avg       0.76      0.75      0.75      1086
weighted avg       0.76      0.75      0.75      1086

  Confusion Matrix:
[[443 100]
 [170 373]]


────────────────────────────────────

# XGBoost

In [2]:
# Arabic Sentiment Analysis — XGBoost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError("Install XGBoost first:  pip install xgboost")

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH = "/content/train_clean.csv"
VAL_PATH   = "/content/val_clean.csv"
TEST_PATH  = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# XGBoost natively supports sparse — no .toarray() needed
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])

n_classes = len(le.classes_)
print(f"Classes : {le.classes_}  →  {list(range(n_classes))}\n")

# 4. MODEL — XGBoost
# tree_method="hist" uses histogram-based splitting — fast on large sparse data.
# scale_pos_weight handles binary imbalance; for multiclass use sample_weight.
print("=" * 60)
print("  ⚡ Training: XGBoost")
print("=" * 60)

objective = "binary:logistic" if n_classes == 2 else "multi:softmax"

clf = XGBClassifier(
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,   # feature fraction per tree
    colsample_bylevel = 0.8,   # feature fraction per level
    reg_alpha         = 0.1,   # L1 regularisation
    reg_lambda        = 1.0,   # L2 regularisation
    objective         = objective,
    num_class         = n_classes if n_classes > 2 else None,
    eval_metric       = "logloss" if n_classes == 2 else "mlogloss",
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = RANDOM_STATE,
    verbosity         = 0,
)

clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  XGBoost  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 10,264
Final feature shape (train) : (5067, 10266)

Classes : [0 1]  →  [0, 1]

  ⚡ Training: XGBoost

────────────────────────────────────────────────────────────
  XGBoost  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7385
  Precision : 0.7435
  Recall    : 0.7385
  F1-Score  : 0.7371

  Classification Report:

              precision    recall  f1-score   support

           0       0.71      0.81      0.76       543
           1       0.78      0.67      0.72       543

    accuracy                           0.74      1086
   macro avg       0.74      0.74      0.74      1086
weighted avg       0.74      0.74      0.74      1086

  Confusion Matrix:
[[440 103]
 [181 362]]


────────────────────────────────────────────────────────────
  XGBoost  |  Te

# LightGBM  

In [3]:
# Arabic Sentiment Analysis — LightGBM

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

try:
    import lightgbm as lgb
except ImportError:
    raise ImportError("Install LightGBM first:  pip install lightgbm")

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH = "/content/train_clean.csv"
VAL_PATH   = "/content/val_clean.csv"
TEST_PATH  = "/content/test_clean.csv"
TEXT_COL      = "clean_text"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# LightGBM handles sparse matrices natively — no .toarray() needed
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])

n_classes = len(le.classes_)
print(f"Classes : {le.classes_}  →  {list(range(n_classes))}\n")

# 4. MODEL — LightGBM
# LightGBM grows trees leaf-wise (best-first) rather than level-wise,
# making it faster and often more accurate than XGBoost on large datasets.
# num_leaves controls model complexity — keep < 2^max_depth.
print("=" * 60)
print("  ⚡ Training: LightGBM")
print("=" * 60)

objective  = "binary"       if n_classes == 2 else "multiclass"
metric     = "binary_logloss" if n_classes == 2 else "multi_logloss"

clf = lgb.LGBMClassifier(
    n_estimators     = 500,
    learning_rate    = 0.05,
    num_leaves       = 63,      # key complexity parameter
    max_depth        = -1,      # no limit — controlled via num_leaves
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,     # L1
    reg_lambda       = 1.0,     # L2
    class_weight     = "balanced",
    objective        = objective,
    metric           = metric,
    n_jobs           = -1,
    random_state     = RANDOM_STATE,
    verbose          = -1,
)

clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  LightGBM  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 10,264
Final feature shape (train) : (5067, 10266)

Classes : [0 1]  →  [0, 1]

  ⚡ Training: LightGBM

────────────────────────────────────────────────────────────
  LightGBM  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.7366
  Precision : 0.7390
  Recall    : 0.7366
  F1-Score  : 0.7360

  Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.79      0.75       543
           1       0.76      0.69      0.72       543

    accuracy                           0.74      1086
   macro avg       0.74      0.74      0.74      1086
weighted avg       0.74      0.74      0.74      1086

  Confusion Matrix:
[[427 116]
 [170 373]]


────────────────────────────────────────────────────────────
  LightGBM  | 